In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

# ---------- Load & split ----------
loader = PyPDFLoader("GK_question_ans.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
text = splitter.split_documents(pages)
chunks = [i.page_content for i in text]

# ---------- Vector store ----------
embedding = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path='./hybrid_RAG')

collection = client.get_or_create_collection(
    name="Data",
    embedding_function=embedding)

# FIX: guard against stale DB — if chunk count doesn't match what's stored,
# wipe and rebuild instead of silently using out-of-date/mismatched data
if collection.count() != len(chunks):
    if collection.count() > 0:
        client.delete_collection("Data")
        collection = client.get_or_create_collection(
            name="Data",
            embedding_function=embedding,
            metadata={"hnsw:space": "cosine"}
        )
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=[i.metadata for i in text]
    )

print("Chunks in collection:", collection.count())

# ---------- BM25 ----------
corpus_m = [i.lower().split() for i in chunks]  # FIX: lowercase consistently with query
token_corpus = BM25Okapi(corpus=corpus_m)

# ---------- Retrieval ----------
def retrive(query: str) -> str:
    query_lower = query.lower()

    result = collection.query(query_texts=[query_lower], n_results=5)
    documents = result['documents'][0]
    distances = result['distances'][0]

    threshold = 0.5  # cosine distance now — tune after checking real numbers
    print("DISTANCES:", distances)

    good_chunks = [doc for dist, doc in zip(distances, documents) if dist < threshold]

    # FIX: was passing raw string (iterated char-by-char); must tokenize
    score = token_corpus.get_scores(query_lower.split())

    def get_nearest_index(scores, k=10):
        index = list(enumerate(score))
        ranked = sorted(index, key=lambda x: x[1], reverse=True)
        return [idx for idx, s in ranked[:k] if s > 0]  # FIX: drop zero-score junk matches

    top_indices = get_nearest_index(score, k=10)
    bm25_chunks = [chunks[i] for i in top_indices]

    # ---------- RRF fusion ----------
    rrf_scores = {}
    for rank, doc in enumerate(good_chunks):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (rank + 60)
    for rank, doc in enumerate(bm25_chunks):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (rank + 60)

    merged = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    top_chunks = [doc for doc, _ in merged[:5]]

    if top_chunks:
        return "\n\n".join(top_chunks)
    return "NOT RELEVANT CONTENT"

# ---------- LLM ----------
key = os.getenv('GROQ_API_KEY')
llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0)

question = "which city called as a city of joy in india "
context = retrive(question)

prompt = f'''You are a reliable assistant. Answer the question using ONLY the content below.
If the content does not contain the answer, respond with exactly: NOT RELEVANT CONTENT

content:
{context}

question: {question}
'''

result = llm.invoke(prompt)
print(result.content)

C:\Users\TOTAN SANTRA\AppData\Local\Temp\ipykernel_6120\791849982.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\PYTHON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6640.23it/s]


Chunks in collection: 70
DISTANCES: [0.4094676375389099, 0.5263389945030212, 0.5371043682098389, 0.6006007194519043, 0.6061879992485046]
Kolkata
